## Lecture1:Colab（無料枠）で試せる、日本語LLM(youri7b)

Pythonを使えば、ChatGPTに代表されるLLM(大規模言語モデル)を動かして動作を確認することもできます（計算機の性能により動作しい場合もある）。



### rinna/youri-7b

rinnaが開発した、meta社のLlama2_7Bをベースに日本語の学習データを用いて継続事前学習を行ったのが、汎用言語モデル「Youri 7B」です。

さらに、汎用言語モデルであるYouri 7Bに、対話形式でユーザーの指示を遂行するような追加学習をしたモデルが「Youri 7B Instruction」です。日本語の一問一答に応える能力が高くベンチマークにおいて高いスコアを達成します。

rinna/youri-7b-instruction-gptq　は、Youri 7B instructionを量子化してコンパクトにしたもので、これならば、日本語の性能がそそこでかつColab無料枠でも動作します。




https://huggingface.co/rinna/youri-7b-instruction-gptq


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("rinna/youri-7b-instruction")
model = AutoModelForCausalLM.from_pretrained("rinna/youri-7b-instruction")

In [ ]:
def get_responce(instruction, input):

  prompt = f"""
  以下は、タスクを説明する指示と、文脈のある入力の組み合わせです。要求を適切に満たす応答を書きなさい。

  ### 指示:
  {instruction}

  ### 入力:
  {input}

  ### 応答:
  """
  token_ids = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt")

  with torch.no_grad():
      output_ids = model.generate(
          input_ids=token_ids.to(model.device),
          max_new_tokens=200,
          do_sample=True,
          temperature=0.5,
          pad_token_id=tokenizer.pad_token_id,
          bos_token_id=tokenizer.bos_token_id,
          eos_token_id=tokenizer.eos_token_id
      )

  output = tokenizer.decode(output_ids.tolist()[0])
  return output

In [ ]:
instruction = "あなたは優秀なAIです。的確に答えてください。"
input = "500円のケーキを３人で食べました、それぞれいくら払うべき？"

res = get_responce(instruction, input)
print(res)

In [ ]:
instruction = "あなたは優秀なAIです。的確に答えてください。"
input = "世界で一番高い山はなんですか？それは東京ドームより高いですか？"

res = get_responce(instruction, input)
print(res)

In [ ]:
instruction = "あなたは優秀なAIです。的確に答えてください。"
input = "清少納言というのは正式な名前か？"
res = get_responce(instruction, input)
print(res)